# Training data chip creation
something descriptive about 300x300 dataset

# Setup
Imports and repo clone.

In [1]:
# !rm -rf ~/.local/lib/python*

In [2]:
# !pip install rioxarray geopandas

In [3]:
import json
import logging
import os
import sys
import time
import shutil
from glob import glob
from pathlib import Path
from contextlib import redirect_stdout, nullcontext
from functools import partial
from collections import Counter
from io import StringIO
import multiprocessing as mp
from multiprocessing import Pool, cpu_count, get_logger

import geopandas as gpd
import numpy as np
import rasterio
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.crs import CRS
from tqdm import tqdm
import subprocess

try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False

In [4]:
# !rm -rf ./lfm

In [5]:
repo_dir = "lfm"

try:
    if not os.path.exists(repo_dir):
        result = subprocess.run(
            ["git", "clone", f"https://github.com/nasa-nccs-hpda/{repo_dir}"],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"Successfully cloned repository to {repo_dir}/")
    else:
        result = subprocess.run(
            ["git", "-C", repo_dir, "pull"],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"Successfully pulled latest changes in {repo_dir}/")
except (subprocess.CalledProcessError, FileNotFoundError) as e:
    print(e)
    print("Git subprocess call failed (known issue in Jupyter notebooks)")
    print("\nPlease run this command in your terminal:")
    print(f"  git clone https://github.com/nasa-nccs-hpda/{repo_dir}")

repoPath = Path(os.getcwd()) / repo_dir
sys.path.append(str(repoPath.parent))

Successfully pulled latest changes in lfm/


In [6]:
result2 = subprocess.run(
    ["git", "-C", repo_dir, "checkout", "develop"],
    check=True,
    capture_output=True,
    text=True
)

In [7]:
from lfm.model.Pipeline import Pipeline
from lfm.model.chip_making.chip_utils import extract_product_id, run_pipeline_for_sample, get_worker_logger, group_cubes_by_tile
from lfm.model.chip_making.chip_constants import PROJECT_DIR, GPKG_PATH, TILE_DB_PATH, ZOOM_LEVEL, CHIP_DIR, LABEL_DIR

In [8]:
logger = get_worker_logger()

# User configuration

In [9]:
output_dir = PROJECT_DIR / "model_inputs/notebook_outputs"
output_dir.mkdir(exist_ok=True)

# Example run: single train sample

### 1. Load training geometries into GeoDataFrame

In [10]:
train_gdf_path = GPKG_PATH
train_gdf = gpd.read_file(GPKG_PATH)
print(f"Loaded train gdf with {len(train_gdf)} entries.")

Loaded train gdf with 674 entries.


### 2. Run datacube pipeline for single sample

In [11]:
entry = train_gdf.iloc[0].to_dict()
entry

{'location': 'M1096558039CE_r7650_c750_input.tif',
 'geometry': <POLYGON ((-11.389 -5.164, -10.396 -5.168, -10.4 -6.158, -11.395 -6.152, -11...>}

In [12]:
train_fn = entry['location']
product_id = extract_product_id(train_fn)
print(f"Product ID: {product_id}")

# Save to datacube subdirectory
train_fn_no_ext = train_fn.replace('.tif', '')
datacube_base_dir = output_dir / "datacubes"
datacube_dir = datacube_base_dir / train_fn_no_ext
datacube_dir.mkdir(exist_ok=True, parents=True)

Product ID: M1096558039CE


In [ ]:
print("Running pipeline...")

pipeline_start = time.time()

cube_files, pipeline_status = run_pipeline_for_sample(
    train_fn=train_fn,
    product_id=product_id,
    geom_bounds=entry['geometry'].bounds,  # (ulLon, lrLat, lrLon, ulLat)
    datacube_dir=datacube_dir,
    TILE_DB_PATH=TILE_DB_PATH,
    logger=logger,
    zoom_level=ZOOM_LEVEL
)
pipeline_end = time.time() - pipeline_start

if not cube_files:
    print("WARNING: no datacubes created.")
if pipeline_status != "success":
    print(f"WARNING: Pipeline status: {pipeline_status}")
else:
    print(f"Pipeline successfully ran for train sample. Number of files created: {len(cube_files)}")
    print(f"Pipeline ran in {pipeline_end} seconds.")

### 3. Match WAC/STATIC files across tile indices
This allows us to merge them into a single chip later. 

In [ ]:
# Step a: group them by LTM tile
cubes_by_tile, _ = group_cubes_by_tile([str(f) for f in cube_files], logger

# Step b: match WAC/STATIC across tiles
wac_files = []
static_files = []

for tile_id, elems in cubes_by_tile.items():
    wac_match = [f for f in elems['wac'] if f"ProdId-{product_id}" in f]
    static_match = [f for f in elems['static']]
    wac_files.append(wac_match[0])
    static_files.append(static_match[0])

### 4. Load reference train chip
This allows us to clip to the proper extent of our labels.

In [ ]:
train_path = CHIP_DIR / train_fn
train_ds = rxr.open_rasterio(train_path)

target_crs = train_ds.rio.crs
target_transform = train_ds.rio.transform()
target_height = train_ds.shape[-2]
target_width = train_ds.shape[-1]
train_bbox = train_ds.rio.bounds()
train_shape = train_ds.shape

## 5. Merge reproject, and clip WAC, STATIC datacubes
This gets them aligned with the original training sample and labels. 

In [ ]:
merge_reproj_result = merge_and_reproject_datasets(
    wac_files=wac_files,
    static_files=static_files,
    target_crs=target_crs,
    target_transform=target_transform,
    target_height=target_height,
    target_width=target_width,
    band_regex=band_regex,
    logger=logger
)
merged_wac, merged_static, wac_band_names, static_band_names, _ = merge_reproj_result

In [ ]:
combined, all_band_names, combine_status = clip_and_combine_datasets(
    merged_wac=merged_wac,
    merged_static=merged_static,
    wac_band_names=wac_band_names,
    static_band_names=static_band_names,
    train_bbox=train_bbox,
    train_shape=train_shape,
    logger=logger
)

## 6. Save chip to file
We will now save our in-memory chip to a file in the output directory.

In [ ]:
chip_output_dir = output_dir / "chips"
chip_output_dir.mkdir(exist_ok=True, parents=True)
output_filename = chip_output_dir / f"{train_fn_no_ext}_wac_static_chip.tif"

write_status = write_chip_to_tif(
    combined=combined,
    output_filename=output_filename,
    all_band_names=all_band_names,
    logger=logger,
    nodata_value=COMMON_NODATA
)

## 7. Copy over matching label
This is so that we have matching image/label pairs for training.

In [ ]:
label_path = next(LABEL_DIR.glob(f"*{product_id}.npz"))
print(f"Found matching label: {label_path}")

label_output_dir = output_dir / "labels"
label_output_dir.mkdir(exist_ok=True, parents=True)
label_basename = f"{train_fn.split("_input")[0]}_label.npz"

shutil.copy(label_path, label_output_dir / label_basename)
label_new_path = next(label_output_dir.glob(label_basename))
if label_new_path: 
    print(f"Label copied successfully!")

## 8. Clean up
Clean our memory by closing datasets.

In [ ]:
combined.close()
merged_wac.close()
merged_static.close()
train_ds.close()